<a href="https://colab.research.google.com/github/Thcastro2004/ECSE551-A2-ML-for-engineers/blob/main/Barnett_Cottereau_Zhang_Assignment2_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Imports

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import numpy as np
import csv
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, Subset, DataLoader
from sklearn.model_selection import train_test_split, KFold
from PIL import Image
import os
import zipfile
from kaggle.api.kaggle_api_extended import KaggleApi
from tqdm import tqdm
import torchvision.models as models
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn.functional as F

## Download Kaggle Dataset

In [ ]:
# Download Kaggle competition data if not already present
data_path = '/content/drive/MyDrive/data'
if not os.path.exists(f'{data_path}/kaggle_challenge/train') or not os.path.exists(f'{data_path}/kaggle_challenge/test'):
    api = KaggleApi()
    api.authenticate()
    os.makedirs(data_path, exist_ok=True)
    
    competition_name = 'ecse-551-assignment-2-task-2'
    api.competition_download_files(competition_name, path=data_path)
    
    zip_path = f'{data_path}/{competition_name}.zip'
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(data_path)
        os.remove(zip_path)

## Dataset Classes

In [ ]:
class TrainingDataset(Dataset):
    def __init__(self, data_path='/content/drive/MyDrive/data'):
        super().__init__()
        self.df = pd.read_csv(f'{data_path}/kaggle_challenge/train_labels.csv')
        self.trainpath = f'{data_path}/kaggle_challenge/train/'
        self.transform = transforms.Compose([
            transforms.RandomApply(nn.ModuleList([transforms.RandomResizedCrop(size=(32,32), scale=(0.8,1))]),p=0.1),
            transforms.RandomApply(nn.ModuleList([transforms.RandomRotation((1,5))]),p=0.1),
            transforms.RandomHorizontalFlip(p=0.1),
            transforms.RandomApply(nn.ModuleList([transforms.ColorJitter((0.7,1),(0.7,1),(0.7,1),(-0.1,0.1))]),p=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
        ])
        self.labels_dict = {
            "truck": 0, "deer": 1, "bird": 2, "frog": 3, "ship": 4,
            "horse": 5, "cat": 6, "dog": 7, "automobile": 8, "airplane": 9
        }
    
    def __getitem__(self, idx):
        img = Image.open(self.trainpath+self.df.at[idx, "id"])
        img = img.convert("RGB")
        img = self.transform(img)
        label = self.labels_dict.get(self.df.at[idx, "label"])
        return img, label
    
    def __len__(self):
        return len(self.df)

In [ ]:
class TestingDataset(Dataset):
    def __init__(self, data_path='/content/drive/MyDrive/data'):
        super().__init__()
        self.df = pd.read_csv(f'{data_path}/kaggle_challenge/train_labels.csv')
        self.trainpath = f'{data_path}/kaggle_challenge/train/'
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
        ])
        self.labels_dict = {
            "truck": 0, "deer": 1, "bird": 2, "frog": 3, "ship": 4,
            "horse": 5, "cat": 6, "dog": 7, "automobile": 8, "airplane": 9
        }
    
    def __getitem__(self, idx):
        img = Image.open(self.trainpath+self.df.at[idx, "id"])
        img = img.convert("RGB")
        img = self.transform(img)
        label = self.labels_dict.get(self.df.at[idx, "label"])
        return img, label
    
    def __len__(self):
        return len(self.df)

In [ ]:
class Test551(Dataset):
    def __init__(self, data_path='/content/drive/MyDrive/data'):
        super().__init__()
        self.df = pd.read_csv(f'{data_path}/kaggle_challenge/sample_submission.csv')
        self.trainpath = f'{data_path}/kaggle_challenge/test/'
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.labels_dict = {
            "truck": 0, "deer": 1, "bird": 2, "frog": 3, "ship": 4,
            "horse": 5, "cat": 6, "dog": 7, "automobile": 8, "airplane": 9
        }
    
    def __getitem__(self, idx):
        img = Image.open(self.trainpath+self.df.at[idx, "id"])
        img = img.convert("RGB")
        img = self.transform(img)
        label = 0
        return img, label
    
    def __len__(self):
        return len(self.df)

In [ ]:
class Task2TrainingDataset(Dataset):
    def __init__(self, data_path='/content/drive/MyDrive/data'):
        super().__init__()
        self.df = pd.read_csv(f'{data_path}/kaggle_challenge/train_labels.csv')
        self.trainpath = f'{data_path}/kaggle_challenge/train/'
        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(size=(32, 32), scale=(0.7, 1.0)),
            transforms.RandomRotation(degrees=15),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.1),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.labels_dict = {
            "truck": 0, "deer": 1, "bird": 2, "frog": 3, "ship": 4,
            "horse": 5, "cat": 6, "dog": 7, "automobile": 8, "airplane": 9
        }
    
    def __getitem__(self, idx):
        img = Image.open(self.trainpath + self.df.at[idx, "id"])
        img = img.convert("RGB")
        img = self.transform(img)
        label = self.labels_dict.get(self.df.at[idx, "label"])
        return img, label
    
    def __len__(self):
        return len(self.df)

In [ ]:
class Task2TestDataset(Dataset):
    def __init__(self, data_path='/content/drive/MyDrive/data'):
        super().__init__()
        self.df = pd.read_csv(f'{data_path}/kaggle_challenge/train_labels.csv')
        self.trainpath = f'{data_path}/kaggle_challenge/train/'
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.labels_dict = {
            "truck": 0, "deer": 1, "bird": 2, "frog": 3, "ship": 4,
            "horse": 5, "cat": 6, "dog": 7, "automobile": 8, "airplane": 9
        }
    
    def __getitem__(self, idx):
        img = Image.open(self.trainpath + self.df.at[idx, "id"])
        img = img.convert("RGB")
        img = self.transform(img)
        label = self.labels_dict.get(self.df.at[idx, "label"])
        return img, label
    
    def __len__(self):
        return len(self.df)

# Task 1 - CNN

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv4 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv5 = nn.Conv2d(128, 256, 4, padding=1)
        self.conv6 = nn.Conv2d(256, 512, 4, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.lin1 = nn.Linear(8192, 4096)
        self.lin2 = nn.Linear(4096, 512)
        self.lin3 = nn.Linear(512, 128)
        self.lin4 = nn.Linear(128, 10)
        self.batchnorm1 = nn.BatchNorm2d(16)
        self.batchnorm2 = nn.BatchNorm2d(32)
        self.batchnorm3 = nn.BatchNorm2d(64)
        self.batchnorm4 = nn.BatchNorm2d(128)
        self.batchnorm5 = nn.BatchNorm2d(256)
        self.batchnorm6 = nn.BatchNorm2d(512)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        x = nn.functional.relu(self.batchnorm1(self.conv1(x)))
        x = nn.functional.relu(self.batchnorm2(self.conv2(x)))
        x = self.pool(x)
        x = nn.functional.relu(self.batchnorm3(self.conv3(x)))
        x = nn.functional.relu(self.batchnorm4(self.conv4(x)))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = nn.functional.relu(self.lin1(x))
        x = self.dropout(x)
        x = nn.functional.relu(self.lin2(x))
        x = self.dropout(x)
        x = nn.functional.relu(self.lin3(x))
        x = self.dropout(x)
        x = self.lin4(x)
        return x

## Task 1 CNN Training

In [ ]:
# Split training data
training_dataset = TrainingDataset()
testing_dataset = TestingDataset()

indices = list(range(len(training_dataset)))
train_indices, test_indices = train_test_split(indices, test_size=0.3, random_state=42)
train_dataset = Subset(training_dataset, train_indices)
validation_dataset = Subset(testing_dataset, train_indices)
final_test_dataset = Subset(testing_dataset, test_indices)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

results = []
kf = KFold(shuffle=True, random_state=42)

for i, (train_idx, test_idx) in enumerate(kf.split(train_indices)):
    print(f"\n{'='*50}")
    print(f"Fold {i+1}")
    print(f"{'='*50}")
    
    cnn = CNN()
    cnn = cnn.to(device)
    cnn.train()
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(cnn.parameters(), 0.0003, weight_decay=0.0001)

    kf_train_dataset = Subset(train_dataset, train_idx)
    kf_train_dataloader = DataLoader(kf_train_dataset, batch_size=64, shuffle=True, num_workers=2)
    kf_test_dataset = Subset(validation_dataset, test_idx)
    kf_test_dataloader = DataLoader(kf_test_dataset, batch_size=64, shuffle=False, num_workers=2)

    training_loss = []
    num_epochs = 5
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        num_batches = 0
        
        pbar = tqdm(kf_train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}", ncols=100)
        for input, label in pbar:
            input, label = input.to(device), label.to(device)
            output = cnn(input)
            loss = loss_function(output, label)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            loss_item = loss.item()
            training_loss.append(loss_item)
            epoch_loss += loss_item
            num_batches += 1
            
            pbar.set_postfix({'loss': f'{loss_item:.4f}'})
        
        avg_epoch_loss = epoch_loss / num_batches
        print(f"Epoch {epoch+1}/{num_epochs} - Average Loss: {avg_epoch_loss:.4f}")

    plt.plot(training_loss)
    plt.xlabel("Batch number")
    plt.ylabel("Loss")
    plt.title(f"Batch-wise Training Loss - Fold {i+1}")
    plt.show()

    # Validation
    print(f"Evaluating fold {i+1}...")
    cnn.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for input, label in tqdm(kf_test_dataloader, desc="Validation", ncols=100):
            input, label = input.to(device), label.to(device)
            output = cnn(input)
            prediction = output.argmax(dim=1)
            correct += (prediction == label).sum().item()
            total += label.size(0)
    
    fold_accuracy = correct / total
    results.append(fold_accuracy)
    print(f"Fold {i+1} Accuracy: {fold_accuracy:.4f}")
    print(f"Current Results: {results}")

print(f"\n{'='*50}")
print(f"Cross-Validation Complete!")
print(f"All Results: {results}")
print(f"Mean Accuracy: {np.mean(results):.4f}")
print(f"{'='*50}")

# Task 2 - Free for All

In [ ]:
def create_resnet_model(num_classes=10):
    model = models.resnet50(weights='IMAGENET1K_V2')
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    return model

def create_efficientnet_model(num_classes=10):
    model = models.efficientnet_b3(weights='IMAGENET1K_V1')
    num_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_features, num_classes)
    return model

def create_vit_model(num_classes=10):
    model = models.vit_b_16(weights='IMAGENET1K_V1')
    num_features = model.heads.head.in_features
    model.heads.head = nn.Linear(num_features, num_classes)
    return model

## Task 2 Training with Cross-Validation

In [ ]:
task2_train_dataset = Task2TrainingDataset()
task2_test_dataset = Task2TestDataset()

# Split into training and test sets
all_indices = list(range(len(task2_train_dataset)))
train_indices, test_indices = train_test_split(all_indices, test_size=0.2, random_state=42)
task2_test_indices = test_indices

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print(f"\nTotal samples: {len(all_indices)}")
print(f"Training samples (for CV): {len(train_indices)}")
print(f"Test samples (held out): {len(test_indices)}")

# Cross-validation on training data only
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []
best_cv_acc = 0.0
best_fold = 0
batch_size = 128

for fold, (cv_train_idx, cv_val_idx) in enumerate(kf.split(train_indices)):
    print(f"\n{'='*60}")
    print(f"Fold {fold+1}/5")
    print(f"{'='*60}")
    
    # Map CV indices back to original dataset indices
    cv_train_indices = [train_indices[i] for i in cv_train_idx]
    cv_val_indices = [train_indices[i] for i in cv_val_idx]
    
    # Create datasets for this fold
    fold_train_subset = Subset(task2_train_dataset, cv_train_indices)
    fold_val_subset = Subset(task2_test_dataset, cv_val_indices)
    
    fold_train_loader = DataLoader(fold_train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
    fold_val_loader = DataLoader(fold_val_subset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    # Create model for this fold
    model_task2 = create_resnet_model(num_classes=10)
    model_task2 = model_task2.to(device)
    
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model_task2.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)
    
    # Training loop for this fold
    num_epochs = 50
    fold_best_val_acc = 0.0
    val_frequency = 2
    
    for epoch in range(num_epochs):
        model_task2.train()
        train_loss = 0.0
        for inputs, labels in tqdm(fold_train_loader, desc=f'Epoch {epoch+1}/{num_epochs}', leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model_task2(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        scheduler.step()
        
        # Validation on CV validation set
        if (epoch + 1) % val_frequency == 0 or epoch == 0 or epoch == num_epochs - 1:
            model_task2.eval()
            val_correct = 0
            val_total = 0
            with torch.no_grad():
                for inputs, labels in tqdm(fold_val_loader, desc='Validation', leave=False):
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model_task2(inputs)
                    _, predicted = torch.max(outputs.data, 1)
                    val_total += labels.size(0)
                    val_correct += (predicted == labels).sum().item()
            
            val_acc = val_correct / val_total
            
            if val_acc > fold_best_val_acc:
                fold_best_val_acc = val_acc
        else:
            val_acc = None
        
        if (epoch + 1) % 10 == 0 or epoch == 0:
            if val_acc is not None:
                print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss/len(fold_train_loader):.4f}, CV Val Acc: {val_acc:.4f}')
            else:
                print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss/len(fold_train_loader):.4f}')
    
    print(f'Fold {fold+1} complete. Best CV validation accuracy: {fold_best_val_acc:.4f}')
    cv_results.append(fold_best_val_acc)
    
    # Save model if this is the best fold so far
    if fold_best_val_acc > best_cv_acc:
        best_cv_acc = fold_best_val_acc
        best_fold = fold + 1
        torch.save(model_task2.state_dict(), '/content/drive/MyDrive/task2_best_model.pth')
        print(f'New best model saved from Fold {best_fold}! CV Val Acc: {best_cv_acc:.4f}')

print(f"\n{'='*60}")
print("Cross-Validation Complete!")
print(f"{'='*60}")
print(f"CV Results: {cv_results}")
print(f"Mean CV Accuracy: {np.mean(cv_results):.4f}")
print(f"Std CV Accuracy: {np.std(cv_results):.4f}")
print(f"Best Fold: {best_fold} with accuracy: {best_cv_acc:.4f}")

# Save test_indices for later evaluation
np.save('/content/drive/MyDrive/task2_test_indices.npy', test_indices)
print(f"\nTest indices saved")
print(f"{'='*60}")

## Task 2 Test Evaluation

In [ ]:
# Load datasets
task2_test_dataset = Task2TestDataset()

# Load test indices that were saved during training
test_indices = np.load('/content/drive/MyDrive/task2_test_indices.npy')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print(f"\n{'='*60}")
print("Evaluating Best Model on Test Set (held out during training)")
print(f"{'='*60}")

# Create test set loader
test_subset = Subset(task2_test_dataset, test_indices)
test_loader = DataLoader(test_subset, batch_size=32, shuffle=False, num_workers=2)

# Load best model and evaluate on test set
model_task2 = create_resnet_model(num_classes=10)
model_task2.load_state_dict(torch.load('/content/drive/MyDrive/task2_best_model.pth'))
model_task2 = model_task2.to(device)
model_task2.eval()

test_correct = 0
test_total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model_task2(inputs)
        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

test_accuracy = test_correct / test_total
print(f"Test Set Accuracy: {test_accuracy:.4f} ({test_correct}/{test_total})")
print(f"{'='*60}")

## Task 2 Kaggle Submission Generation

In [ ]:
# Load the test dataset for Kaggle submission
kaggle_test_dataset = Test551()
kaggle_test_loader = DataLoader(kaggle_test_dataset, batch_size=64, shuffle=False, num_workers=2)

# Load best model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_task2 = create_resnet_model(num_classes=10)
model_task2.load_state_dict(torch.load('/content/drive/MyDrive/task2_best_model.pth'))
model_task2 = model_task2.to(device)
model_task2.eval()

# Reverse label dictionary for converting predictions to labels
idx_to_label = {0: "truck", 1: "deer", 2: "bird", 3: "frog", 4: "ship",
                5: "horse", 6: "cat", 7: "dog", 8: "automobile", 9: "airplane"}

# Generate predictions
predictions = []
ids = []

with torch.no_grad():
    for inputs, _ in tqdm(kaggle_test_loader, desc='Generating predictions'):
        inputs = inputs.to(device)
        outputs = model_task2(inputs)
        _, predicted = torch.max(outputs.data, 1)
        predictions.extend(predicted.cpu().numpy())

# Get IDs from the dataset
for idx in range(len(kaggle_test_dataset)):
    ids.append(kaggle_test_dataset.df.at[idx, 'id'])

# Convert predictions to labels
labels = [idx_to_label[pred] for pred in predictions]

# Create submission DataFrame
submission_df = pd.DataFrame({'id': ids, 'label': labels})

# Save submission file
submission_path = '/content/drive/MyDrive/task2_submission.csv'
submission_df.to_csv(submission_path, index=False)
print(f"Submission file saved to {submission_path}")
print(f"Total predictions: {len(predictions)}")